In [1]:
input_1 = 'file_input_uploads/Factordata-20241230-182848.zip'

In [2]:
!pip install openpyxl
!pip install matplotlib-venn
!pip install imbalanced-learn
!pip install TA-Lib


import os
print(os.listdir("file_input_uploads"))


[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
  Using cached ta_lib-0.6.0.tar.gz (371 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for TA-Lib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [38 lines of output]
      <string>:75: UserWarning: Cannot find ta-lib library, installation may fail.
      /tmp/pip-build-env-11cz7cfo/overlay/lib/python3.9/site-packages/setuptools/config/_apply_pyprojecttoml.py:81: SetuptoolsWarning: `install_requires` overwritten in `pyproject.toml` (dependencie

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import os
import json
from datetime import datetime
import logging
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

class Config:
    def __init__(self):
        # Paths
        self.FACTOR_DATA_PATH = 'Factordata'
        self.OUTPUT_DIR = os.path.join(self.FACTOR_DATA_PATH, 'analysis_output')
        os.makedirs(self.OUTPUT_DIR, exist_ok=True)

        # Analysis parameters
        self.RANDOM_STATE = 42
        self.N_ESTIMATORS = 200
        self.FACTOR_COLUMNS = [
            'Alpha..annualisiert.',
            'Value.Growth',
            'Small.Large',
            'Momentum',
            'Volatility'
        ]

        # LSTM parameters
        self.SEQUENCE_LENGTH = 5
        self.LSTM_UNITS = 64
        self.DROPOUT_RATE = 0.2
        self.EPOCHS = 50
        self.BATCH_SIZE = 32
        self.VALIDATION_SPLIT = 0.2
        self.ENSEMBLE_THRESHOLD = 0.7

class FactorAnomalyAnalyzer:
    def __init__(self, config):
        self.config = config
        self.logger = self._setup_logger()
        self.lstm_models = {}
        self.validation_metrics = {}

    def _setup_logger(self):
        logger = logging.getLogger('factor_anomaly_analysis')
        logger.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

        # File handler
        log_file = os.path.join(self.config.OUTPUT_DIR, 
                                 f'factor_analysis_log_{datetime.now():%Y%m%d_%H%M}.log')
        fh = logging.FileHandler(log_file)
        fh.setFormatter(formatter)
        logger.addHandler(fh)

        # Console handler
        ch = logging.StreamHandler()
        ch.setFormatter(formatter)
        logger.addHandler(ch)

        return logger

    def _build_lstm_model(self, n_features: int, factor: str):
        model = Sequential([
            LSTM(self.config.LSTM_UNITS, 
                 input_shape=(self.config.SEQUENCE_LENGTH, n_features), 
                 return_sequences=True),
            Dropout(self.config.DROPOUT_RATE),
            LSTM(self.config.LSTM_UNITS // 2),
            Dropout(self.config.DROPOUT_RATE),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        self.lstm_models[factor] = model

    def _prepare_sequences(self, features: np.ndarray, target: np.ndarray):
        X, y = [], []
        for i in range(len(features) - self.config.SEQUENCE_LENGTH):
            X.append(features[i:(i + self.config.SEQUENCE_LENGTH)])
            y.append(target[i + self.config.SEQUENCE_LENGTH])
        return np.array(X), np.array(y)

    def validate_anomalies(self, predictions, true_values, factor):
        try:
            rolling_std = pd.Series(true_values).rolling(window=20).std()
            volatility_threshold = rolling_std.mean() + 2 * rolling_std.std()
            true_anomalies = (abs(true_values) > volatility_threshold).astype(int)

            metrics = {
                'temporal_consistency': self._calculate_temporal_consistency(predictions),
                'volatility_alignment': np.mean(predictions == true_anomalies),
                'cluster_score': self._calculate_cluster_score(predictions),
                'persistence_score': self._calculate_persistence_score(predictions, true_values)
            }

            self.validation_metrics[factor] = metrics
            return metrics

        except Exception as e:
            self.logger.error(f"Error in validation for {factor}: {e}")
            return {}

    def _calculate_temporal_consistency(self, predictions, window=5):
        rolling_sum = pd.Series(predictions).rolling(window=window).sum()
        return np.mean(rolling_sum <= window / 2)

    def _calculate_cluster_score(self, predictions):
        clusters = []
        current_cluster = []

        for i, pred in enumerate(predictions):
            if pred == 1:
                current_cluster.append(i)
            elif current_cluster:
                clusters.append(current_cluster)
                current_cluster = []

        if current_cluster:
            clusters.append(current_cluster)

        cluster_sizes = [len(c) for c in clusters] if clusters else [0]
        return 1 - (np.std(cluster_sizes) / (np.mean(cluster_sizes) + 1e-10))

    def _calculate_persistence_score(self, predictions, values, window=3):
        score = 0
        for i in range(len(predictions) - window):
            if predictions[i] == 1:
                future_values = values[i:i + window]
                baseline = np.mean(values[max(0, i - window):i])
                deviation = np.abs(future_values - baseline)
                score += np.mean(deviation > np.std(values))
        return score / (sum(predictions) + 1e-10)

    def create_features(self, data: pd.DataFrame):
        try:
            economic_cols = [col for col in data.columns if col not in self.config.FACTOR_COLUMNS]

            features = data[economic_cols].copy()
            for col in features.columns:
                features[col] = features[col].fillna(method='ffill').fillna(method='bfill').fillna(features[col].mean())

            for col in economic_cols:
                if col in features.columns:
                    features[f'{col}_rolling_mean_3'] = features[col].rolling(window=3).mean()
                    features[f'{col}_rolling_std_3'] = features[col].rolling(window=3).std()

            features = features.fillna(method='ffill').fillna(method='bfill')
            features['month'] = data.index.month
            features['year'] = data.index.year
            features = features.replace([np.inf, -np.inf], np.nan).fillna(method='bfill').fillna(method='ffill')
            return features

        except Exception as e:
            self.logger.error(f"Error creating features: {e}")
            raise

    def detect_anomalies(self, data: pd.DataFrame, factor: str):
        try:
            features_df = self.create_features(data)
            target = data[factor].values

            features_df[f'{factor}_rolling_mean'] = pd.Series(target).rolling(window=5).mean()
            features_df[f'{factor}_rolling_std'] = pd.Series(target).rolling(window=5).std()

            feature_scaler = StandardScaler()
            target_scaler = StandardScaler()

            features_scaled = feature_scaler.fit_transform(features_df)
            target_scaled = target_scaler.fit_transform(target.reshape(-1, 1)).ravel()

            n_iterations = 5
            if_predictions_ensemble = []
            if_scores_ensemble = []

            for i in range(n_iterations):
                sample_idx = np.random.choice(len(features_scaled), size=len(features_scaled), replace=True)

                iso_forest = IsolationForest(
                    n_estimators=self.config.N_ESTIMATORS,
                    contamination='auto',
                    bootstrap=True,
                    max_samples='auto',
                    random_state=self.config.RANDOM_STATE + i,
                    n_jobs=-1
                )

                if_predictions_ensemble.append(iso_forest.fit_predict(features_scaled[sample_idx]))
                if_scores_ensemble.append(iso_forest.score_samples(features_scaled[sample_idx]))

            if_predictions = np.mean([pred == -1 for pred in if_predictions_ensemble], axis=0) > 0.5
            if_scores = np.mean(if_scores_ensemble, axis=0)

            if_scores = (if_scores - np.min(if_scores)) / (np.max(if_scores) - np.min(if_scores) + 1e-10)

            X_seq, y_seq = self._prepare_sequences(features_scaled, target_scaled)
            model_key = f"lstm_{factor}"

            if model_key not in self.lstm_models:
                self._build_lstm_model(features_scaled.shape[1], model_key)

            self.lstm_models[model_key].fit(
                X_seq, y_seq,
                epochs=self.config.EPOCHS,
                batch_size=self.config.BATCH_SIZE,
                validation_split=self.config.VALIDATION_SPLIT,
                callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
                verbose=0
            )

            lstm_pred = self.lstm_models[model_key].predict(X_seq, verbose=0).ravel()
            lstm_scores = np.abs(y_seq - lstm_pred)
            lstm_scores = (lstm_scores - np.min(lstm_scores)) / (np.max(lstm_scores) - np.min(lstm_scores) + 1e-10)

            padded_lstm_scores = np.zeros(len(data))
            padded_lstm_scores[self.config.SEQUENCE_LENGTH:] = lstm_scores

            ensemble_scores = 0.5 * (1 - if_scores) + 0.5 * padded_lstm_scores
            ensemble_predictions = (ensemble_scores > self.config.ENSEMBLE_THRESHOLD).astype(int)

            validation_metrics = self.validate_anomalies(ensemble_predictions, target, factor)
            self.logger.info(f"Validation metrics for {factor}: {validation_metrics}")

            return ensemble_predictions, if_scores, padded_lstm_scores, ensemble_scores

        except Exception as e:
            self.logger.error(f"Error in detect_anomalies for {factor}: {e}")
            raise

    def analyze_all_files(self):
        excel_files = [f for f in os.listdir(self.config.FACTOR_DATA_PATH) if f.endswith(('.xlsx', '.xls'))]

        for file in excel_files:
            try:
                self.logger.info(f"Processing file: {file}")
                file_path = os.path.join(self.config.FACTOR_DATA_PATH, file)
                factor_data = pd.read_excel(file_path, engine='openpyxl')

                if 'Date' not in factor_data.columns:
                    self.logger.error(f"No Date column found in {file}")
                    continue

                factor_data['Date'] = pd.to_datetime(factor_data['Date'])
                factor_data.set_index('Date', inplace=True)

                for factor in self.config.FACTOR_COLUMNS:
                    if factor in factor_data.columns:
                        self.logger.info(f"Analyzing factor: {factor}")
                        try:
                            predictions, if_scores, lstm_scores, ensemble_scores = self.detect_anomalies(factor_data, factor)
                            self.logger.info(f"Detected anomalies in {factor}")
                        except Exception as e:
                            self.logger.error(f"Error analyzing factor {factor}: {e}")
                            continue
            except Exception as e:
                self.logger.error(f"Error analyzing file {file}: {e}")

if __name__ == "__main__":
    config = Config()
    analyzer = FactorAnomalyAnalyzer(config)
    analyzer.analyze_all_files()


2025-01-14 09:00:09.105605: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-14 09:00:09.110101: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-14 09:00:09.144619: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-14 09:00:09.144739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-14 09:00:09.145768: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b0d79e99-778c-4964-b163-e34d369ad413' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>